# ESP32-LLM Training Pipeline on Google Colab
Train Quantization-Aware Ternary (1.58-bit) LLM models (`Micro-LM-Pico`, `Micro-LM-Ultra`, `Micro-LM-S3-Large`) and export checkpoints & C binaries directly to Google Drive.

In [ ]:
# 1. Mount Google Drive & Set Sync Destination
import os
from google.colab import drive

drive.mount('/content/drive')

GDRIVE_OUT_DIR = '/content/drive/MyDrive/esp32_llm_checkpoints'
os.makedirs(GDRIVE_OUT_DIR, exist_ok=True)
print(f'Google Drive mounted! Checkpoints will sync to: {GDRIVE_OUT_DIR}')

In [ ]:
# 2. Check GPU & Install Dependencies
!nvidia-smi
!pip install -q datasets tqdm torch

In [ ]:
# 3. Setup Project Code & Clone / Sync Repository
import os
%cd /content
if not os.path.exists('micro-lm'):
    !git clone https://github.com/adesgautam/micro-lm.git micro-lm
%cd /content/micro-lm
!git pull

In [ ]:
# 4. Prepare Clean Curated Low-Entropy Corpus
import os

if not os.path.exists('datasets/raw/lyrics_corpus.txt'):
    print('Preparing clean curated corpus from HuggingFace...')
    !python scripts/prepare_clean_corpus.py

print('Dataset ready!')

In [ ]:
# 5. Define GDrive Sync Helper
import shutil
import glob

def sync_to_gdrive():
    print(f'Syncing checkpoints and binaries to GDrive ({GDRIVE_OUT_DIR})...')
    if os.path.exists('checkpoints'):
        gdrive_ckpt = os.path.join(GDRIVE_OUT_DIR, 'checkpoints')
        shutil.copytree('checkpoints', gdrive_ckpt, dirs_exist_ok=True)
    
    gdrive_fw = os.path.join(GDRIVE_OUT_DIR, 'firmware_binaries')
    os.makedirs(gdrive_fw, exist_ok=True)
    for bin_file in glob.glob('firmware/src/*.bin'):
        shutil.copy(bin_file, gdrive_fw)
        
    print(f'Sync complete! Files backed up in {GDRIVE_OUT_DIR}')

sync_to_gdrive()

In [ ]:
# 6. Train Teacher: Micro-LM-Ultra (11.4M params) -> Target PPL ~8-12
!python scripts/train_qat.py --config micro_lm_ultra --epochs 40 --gdrive_dir $GDRIVE_OUT_DIR
sync_to_gdrive()

In [ ]:
# 7. Train Distilled Student: Micro-LM-Pico (191K params) with Knowledge Distillation
!python scripts/train_qat.py --config micro_lm_pico --epochs 60 --teacher checkpoints/Micro-LM-Ultra/Micro-LM-Ultra_qat.pth --kd_alpha 0.6 --kd_temp 2.0 --gdrive_dir $GDRIVE_OUT_DIR
sync_to_gdrive()

In [ ]:
# 8. Train Micro-LM-S3-Large (26.2M params for ESP32-S3 with 8MB PSRAM) -> Target PPL ~5-8
!python scripts/train_qat.py --config micro_lm_s3_large --epochs 30 --gdrive_dir $GDRIVE_OUT_DIR
sync_to_gdrive()

In [ ]:
# 9. Final Sync Summary
sync_to_gdrive()
print('ALL TRAINING JOBS AND EXPORTS COMPLETED AND SAVED TO GDRIVE!')